In [1]:
!pip -q install scanpy humanize anndata

In [2]:
!pip install git+https://github.com/gillislab/pyMN#egg=pymetaneighbor

  Cloning https://github.com/gillislab/pyMN to /tmp/pip-install-0yomcpi7/pymetaneighbor_8d015d92288c47d7acef635aa7a43a2b
  Running command git clone --filter=blob:none --quiet https://github.com/gillislab/pyMN /tmp/pip-install-0yomcpi7/pymetaneighbor_8d015d92288c47d7acef635aa7a43a2b
  Resolved https://github.com/gillislab/pyMN to commit 882b54512347be66000f7909271fd8e0cb7def39
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pymetaneighbor: filename=pyMetaNeighbor-0.1.0-py3-none-any.whl size=29125 sha256=34dc93a00eafa20c649fa94d7266d846077c8c373c706a81f24dfb215a974232
  Stored in directory: /tmp/pip-ephem-wheel-cache-tumi5_xl/wheels/81/4f/2f/ceba31de0b80b66b66844e8aa5fa3604b0f9754c5331b4a80e
  Created wheel for upsetplot: filename=upsetplot-0.9.0-py3-none-any.whl size=24864 sha256=d0b8dd393b3b16148689f4c029d7103963eb79db82bd93149febec91189bf

In [3]:
import numpy as np
import pandas as pd
import scanpy as sc
import pymn
import anndata as ad
import scipy.sparse as sp
import time
import os
import gc
import sys
import re
import resource
import time
import datetime
import multiprocessing

In [4]:
A1_h5ad = "/sbgenomics/project-files/GEN_A1/GEN_A1_rc2.h5ad"
A2_h5ad = "/sbgenomics/project-files/GEN_A2/250917_GEN_A2/GEN_A2_rc2.h5ad"

In [9]:
target_h5ad = "/sbgenomics/project-files/GEN_A3/260528_GEN_A3/GEN_A3_PFC_pass2.h5ad"  #GEN_A3_PFC_pass2.h5ad  GEN_A3_PMC_pass2.h5ad  GEN_A3_PVC_pass2.h5ad

In [6]:
# target_h5ad is now passed as the first command-line argument
# Example: python script.py /path/to/file.h5ad
if len(sys.argv) < 2:
    raise SystemExit(f"Usage: {sys.argv[0]} <target_h5ad>")
target_h5ad = sys.argv[1]

In [10]:
internal_dataset_study_id = re.search(r'GEN_\w+(?=_pass)', target_h5ad).group()
print(internal_dataset_study_id) 

GEN_A3_PFC


In [11]:
result_folder = "/sbgenomics/output-files/"+ internal_dataset_study_id + "_vrs_GEN_A1-2.cortex.subclass." + str(round(time.time()))

In [12]:
os.mkdir(result_folder)
print(result_folder)

/sbgenomics/output-files/GEN_A3_PFC_vrs_GEN_A1-2.cortex.subclass.1780583145


In [13]:
start_time = time.time()

In [14]:
internal_dataset = sc.read_h5ad(target_h5ad, backed="r")  # Open in read mode without loading everything

In [15]:
internal_dataset.obs

,Batch,rep,set,Source,brain_region,sample,individualID,libraryID,final_filt,Channel,...,mito_genes,mito_ribo,ribo_genes,apoptosis,class,subclass,subtype,doublet_score,pred_dbl,demux_type
barcodekey,,,,,,,,,,,,,,,,,,,,,
AAACCCAAGACGGTTG-PD-Set11-C1,PD_Set11_C1_cDNA,C1,PD_Set11_C,UMBEB,PFC,PM-UM_BEB20014-BLM0-PFC-RSN,PM-UM_BEB20014,PM-UM_BEB20014_1,True,PM-UM_BEB20014_1,...,-0.932769,-0.149385,-0.288391,-0.107923,6,6_2,6_2_2,0.028823,False,singlet
AAACCCAAGGATTCAA-PD-Set11-C1,PD_Set11_C1_cDNA,C1,PD_Set11_C,UMBEB,PFC,PM-UM_BEB20014-BLM0-PFC-RSN,PM-UM_BEB20014,PM-UM_BEB20014_1,True,PM-UM_BEB20014_1,...,-0.604091,-0.197429,-0.072877,0.280232,7,7_1,7_1_2,0.005720,False,singlet
AAACCCACACAACGAG-PD-Set11-C1,PD_Set11_C1_cDNA,C1,PD_Set11_C,UMBEB,PFC,PM-UM_BEB20014-BLM0-PFC-RSN,PM-UM_BEB20014,PM-UM_BEB20014_1,True,PM-UM_BEB20014_1,...,-0.613243,0.236666,-0.162485,-0.175774,1,1_1,1_1_1,0.039977,False,singlet
AAACCCAGTCCGACGT-PD-Set11-C1,PD_Set11_C1_cDNA,C1,PD_Set11_C,UMBEB,PFC,PM-UM_BEB20014-BLM0-PFC-RSN,PM-UM_BEB20014,PM-UM_BEB20014_1,True,PM-UM_BEB20014_1,...,-0.507562,0.048477,-0.566611,-0.235109,3,3_5,3_5_1,0.041200,False,singlet
AAACCCATCCGTGTGG-PD-Set11-C1,PD_Set11_C1_cDNA,C1,PD_Set11_C,UMBEB,PFC,PM-UM_BEB20014-BLM0-PFC-RSN,PM-UM_BEB20014,PM-UM_BEB20014_1,True,PM-UM_BEB20014_1,...,0.197389,0.079662,-0.066546,0.179350,3,3_1,3_1_1,0.009647,False,singlet
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGGTCACCTTC-PD-set5-2,PD_set5_2_cDNA,2,PD_set5_,MSSM,PFC,PM-MS_114382-BLM0-PFC-RSN,PM-MS_114382,PM-MS_114382_2,True,PM-MS_114382_2,...,0.135056,0.089221,-0.303019,-0.074953,1,1_1,1_1_1,0.044354,False,singlet
TTTGTTGTCAAATGAG-PD-set5-2,PD_set5_2_cDNA,2,PD_set5_,MSSM,PFC,PM-MS_840752-BLM0-PFC-RSN,PM-MS_840752,PM-MS_840752_2,True,PM-MS_840752_2,...,-0.835084,-0.021502,-0.034220,0.050677,1,1_1,1_1_1,0.070812,False,singlet
TTTGTTGTCATCGTAG-PD-set5-2,PD_set5_2_cDNA,2,PD_set5_,MSSM,PFC,PM-MS_840752-BLM0-PFC-RSN,PM-MS_840752,PM-MS_840752_2,True,PM-MS_840752_2,...,-0.340419,0.112971,-0.188896,0.034747,1,1_2,1_2_1,0.026223,False,singlet


In [16]:
## Create a new AnnData object with only the raw matrix
internal_dataset = sc.AnnData(internal_dataset.raw.X, obs=internal_dataset.obs, var=internal_dataset.var)
internal_dataset = internal_dataset.to_memory()

In [17]:
internal_dataset

AnnData object with n_obs × n_vars = 549047 × 35608
    obs: 'Batch', 'rep', 'set', 'Source', 'brain_region', 'sample', 'individualID', 'libraryID', 'final_filt', 'Channel', 'n_genes', 'n_counts', 'percent_mito', 'passed_qc', 'G1/S', 'G2/M', 'cycle_diff', 'cycling', 'predicted_phase', 'female_score', 'male_score', 'gender_score', 'predicted_gender', 'mito_genes', 'mito_ribo', 'ribo_genes', 'apoptosis', 'class', 'subclass', 'subtype', 'doublet_score', 'pred_dbl', 'demux_type'
    var: 'gene_symbols', 'feature_types', 'gene_id', 'gene_name', 'gene_type', 'gene_chrom', 'gene_start', 'gene_end', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features', 'ribo', 'mito', 'protein_coding', 'mitocarta', 'robust_protein_coding', 'robust_protein_coding_autosome', 'mean', 'bins'

In [21]:
internal_dataset.var = internal_dataset.var.set_index("gene_id")

In [22]:
# Create cell_type column
internal_dataset.obs['cell_type'] = internal_dataset.obs['subtype'] #or class
internal_dataset.obs['level_1'] = internal_dataset.obs['class'] #or class
internal_dataset.obs['level_2'] = internal_dataset.obs['subclass']
internal_dataset.obs['level_3'] = internal_dataset.obs['subtype'] #or class
internal_dataset.obs['study_id'] = internal_dataset_study_id

In [23]:
def load_h5ad_sample(h5ad_path, study_id, sample_fraction=1.0, random_seed=42):
    """
    Load h5ad file with optional random sampling of cells.
    
    Parameters:
    -----------
    h5ad_path : str
        Path to the h5ad file
    study_id : str
        Study ID to assign to obs['study_id']
    sample_fraction : float, default=1.0
        Fraction of cells to load (0.0 to 1.0). Use 1.0 to load all cells.
    random_seed : int, default=42
        Random seed for reproducibility. Set to None for no seed.
    
    Returns:
    --------
    AnnData object with cells loaded into memory
    """
    # Open in read mode without loading everything
    adata = sc.read_h5ad(h5ad_path, backed="r")
    
    # Get total number of cells
    n_cells = adata.shape[0]
    
    if sample_fraction < 1.0:
        # Randomly sample cells
        if random_seed is not None:
            np.random.seed(random_seed)
        sample_size = int(n_cells * sample_fraction)
        random_indices = np.random.choice(n_cells, size=sample_size, replace=False)
        random_indices = np.sort(random_indices)
        
        # Load only the sampled cells
        adata = sc.AnnData(adata.raw.X[random_indices, :], 
                          obs=adata.obs.iloc[random_indices], 
                          var=adata.var)
    else:
        # Load all cells
        adata = sc.AnnData(adata.raw.X, obs=adata.obs, var=adata.var)
    
    adata = adata.to_memory()
    adata.var = adata.var.set_index("gene_id")
    adata.obs['study_id'] = study_id
    
    return adata

In [24]:
GEN_A1_sample_fraction = 1
A1 = load_h5ad_sample(A1_h5ad, "GEN_A1", sample_fraction=GEN_A1_sample_fraction)

In [25]:
A2 = load_h5ad_sample(A2_h5ad, "GEN_A2", sample_fraction=1)

In [26]:
A1

AnnData object with n_obs × n_vars = 374028 × 35766
    obs: 'Batch', 'prep', 'rep', 'set', 'final_filt', 'individualID', 'Source', 'libraryID', 'Channel', 'n_genes', 'n_counts', 'percent_mito', 'passed_qc', 'G1/S', 'G2/M', 'cycle_diff', 'cycling', 'predicted_phase', 'female_score', 'male_score', 'gender_score', 'predicted_gender', 'mito_genes', 'mito_ribo', 'ribo_genes', 'apoptosis', 'class', 'subclass', 'subtype', 'doublet_score', 'pred_dbl', 'demux_type', 'study_id'
    var: 'gene_symbols', 'feature_types', 'gene_name', 'gene_type', 'gene_chrom', 'gene_start', 'gene_end', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features', 'ribo', 'mito', 'protein_coding', 'mitocarta', 'robust_protein_coding', 'robust_protein_coding_autosome', 'mean', 'bins'

In [27]:
A2

AnnData object with n_obs × n_vars = 73606 × 27203
    obs: 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'QC_pass', 'pool', 'set', 'rep', 'Source', 'SubID_vs', 'individualID', 'Channel', 'n_counts', 'percent_mito', 'passed_qc', 'G1/S', 'G2/M', 'cycle_diff', 'cycling', 'predicted_phase', 'female_score', 'male_score', 'gender_score', 'predicted_gender', 'mito_genes', 'mito_ribo', 'ribo_genes', 'apoptosis', 'class', 'subclass', 'subtype', 'doublet_score', 'pred_dbl', 'demux_type', 'study_id'
    var: 'gene_symbols', 'feature_types', 'gene_name', 'mito', 'gene_type', 'gene_chrom', 'gene_start', 'gene_end', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features', 'ribo', 'protein_coding', 'mitocarta', 'robust_protein_coding', 'robust_protein_co

In [29]:
h5ad_inputs = pd.DataFrame({
    "h5ad_var": ["target_h5ad", "A1_h5ad", "A2_h5ad"],
    "dataset_name": [internal_dataset_study_id, "GEN_A1", "GEN_A2"],
    "sample_fraction": [1.0, GEN_A1_sample_fraction, 1.0],
    "h5ad_path": [target_h5ad, A1_h5ad, A2_h5ad],
})

h5ad_inputs.to_csv(os.path.join(result_folder, "h5ad_inputs.csv"), index=False)
print(h5ad_inputs)

      h5ad_var dataset_name  sample_fraction  \
0  target_h5ad   GEN_A3_PFC             1.00   
1      A1_h5ad       GEN_A1             0.05   
2      A2_h5ad       GEN_A2             1.00   

                                           h5ad_path  
0  /sbgenomics/project-files/GEN_A3/260528_GEN_A3...  
1   /sbgenomics/project-files/GEN_A1/GEN_A1_rc2.h5ad  
2  /sbgenomics/project-files/GEN_A2/250917_GEN_A2...  


In [32]:
A1.obs['cell_type'] = A1.obs['subclass'] #or class
A2.obs['cell_type'] = A2.obs['subclass'] #or class


In [33]:
merged=ad.concat([internal_dataset, A1, A2], join="inner")

In [34]:
merged.obs.index = merged.obs.index.to_numpy(dtype="str")

In [35]:
pymn.variableGenes(merged, study_col='study_id')

In [36]:
merged = merged[:, merged.var.highly_variable]

In [37]:
#use numpy types instead of pandas
merged.var.highly_variable = merged.var.highly_variable.to_numpy(dtype="bool")
merged.var.index = merged.var.index.to_numpy(dtype="str")

/opt/conda/lib/python3.11/site-packages/pandas/core/generic.py:6234: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  self[name] = value


In [38]:
#make cell indices unique
merged.obs_names_make_unique()

In [39]:
merged.obs['cell_type'] = merged.obs['cell_type'].to_numpy(dtype="str")
merged.obs['study_id'] = merged.obs['study_id'].to_numpy(dtype="str")
merged.obs.index = merged.obs.index.to_numpy(dtype="str")

In [40]:
print("Before running metaneighbor all vs all")

Before running metaneighbor all vs all


In [41]:
#%%time
#run
pymn.MetaNeighborUS(merged,
                    study_col='study_id',
                    ct_col='cell_type',
                    fast_version=True, symmetric_output=True)

/opt/conda/lib/python3.11/site-packages/pymn/MetaNeighborUS.py:62: FutureWarning: The method obs_keys is deprecated and will be removed in the future. Use obs instead of obs_keys. (e.g. `k in adata.obs` or `str(adata.obs.columns.tolist())`)
  assert study_col in adata.obs_keys(), "Study Col not in adata"
/opt/conda/lib/python3.11/site-packages/pymn/MetaNeighborUS.py:63: FutureWarning: The method obs_keys is deprecated and will be removed in the future. Use obs instead of obs_keys. (e.g. `k in adata.obs` or `str(adata.obs.columns.tolist())`)
  assert ct_col in adata.obs_keys(), "Cluster Col not in adata"
/opt/conda/lib/python3.11/site-packages/pymn/MetaNeighborUS.py:73: FutureWarning: The method var_keys is deprecated and will be removed in the future. Use var instead of var_keys. (e.g. `k in adata.var` or `str(adata.var.columns.tolist())`)
  var_genes in adata.var_keys()


In [42]:
print("After running metaneighbor all vs all")
aurocs = merged.uns["MetaNeighborUS"]
aurocs.to_csv(result_folder + "/aurocs_full.csv.gz", compression="gzip")

After running metaneighbor all vs all


In [43]:
#seems to crash here, not sure why - moved to python script
#run 1 vs best
pymn.MetaNeighborUS(merged,
                    study_col='study_id',
                    ct_col='cell_type', one_vs_best=True,
                    fast_version=True, symmetric_output=True)

/opt/conda/lib/python3.11/site-packages/pymn/MetaNeighborUS.py:62: FutureWarning: The method obs_keys is deprecated and will be removed in the future. Use obs instead of obs_keys. (e.g. `k in adata.obs` or `str(adata.obs.columns.tolist())`)
  assert study_col in adata.obs_keys(), "Study Col not in adata"
/opt/conda/lib/python3.11/site-packages/pymn/MetaNeighborUS.py:63: FutureWarning: The method obs_keys is deprecated and will be removed in the future. Use obs instead of obs_keys. (e.g. `k in adata.obs` or `str(adata.obs.columns.tolist())`)
  assert ct_col in adata.obs_keys(), "Cluster Col not in adata"
/opt/conda/lib/python3.11/site-packages/pymn/MetaNeighborUS.py:73: FutureWarning: The method var_keys is deprecated and will be removed in the future. Use var instead of var_keys. (e.g. `k in adata.var` or `str(adata.var.columns.tolist())`)
  var_genes in adata.var_keys()


In [44]:
aurocs = merged.uns["MetaNeighborUS_1v1"]
aurocs.to_csv(result_folder + "/aurocs_1v1.csv.gz", compression="gzip")

In [45]:
cell_counts = merged.obs.groupby("study_id").size()
cell_counts.to_csv(result_folder + "/cell_study_counts.csv")
cell_type_counts = merged.obs[["study_id", "cell_type"]].drop_duplicates().groupby("study_id").size()
cell_type_counts.to_csv(result_folder + "/cell_type_per_study_counts.csv")

In [46]:
for set_threshold in [0.8, 0.9, 0.95, 0.99, 0.999]:
    print(set_threshold)
    pymn.topHits(merged, threshold=set_threshold)
    tophit_table = merged.uns['MetaNeighborUS_topHits']
    tophit_table.to_csv(result_folder + "/top_hits."+str(set_threshold)+".csv")

0.8
0.9
0.95


/opt/conda/lib/python3.11/site-packages/pymn/topHits.py:33: FutureWarning: The method uns_keys is deprecated and will be removed in the future. Use uns instead of uns_keys. (e.g. `k in adata.uns` or `sorted(adata.uns)`)
  mn_key in adata.uns_keys()
/opt/conda/lib/python3.11/site-packages/pymn/topHits.py:33: FutureWarning: The method uns_keys is deprecated and will be removed in the future. Use uns instead of uns_keys. (e.g. `k in adata.uns` or `sorted(adata.uns)`)
  mn_key in adata.uns_keys()
/opt/conda/lib/python3.11/site-packages/pymn/topHits.py:33: FutureWarning: The method uns_keys is deprecated and will be removed in the future. Use uns instead of uns_keys. (e.g. `k in adata.uns` or `sorted(adata.uns)`)
  mn_key in adata.uns_keys()


0.99
0.999


/opt/conda/lib/python3.11/site-packages/pymn/topHits.py:33: FutureWarning: The method uns_keys is deprecated and will be removed in the future. Use uns instead of uns_keys. (e.g. `k in adata.uns` or `sorted(adata.uns)`)
  mn_key in adata.uns_keys()
/opt/conda/lib/python3.11/site-packages/pymn/topHits.py:33: FutureWarning: The method uns_keys is deprecated and will be removed in the future. Use uns instead of uns_keys. (e.g. `k in adata.uns` or `sorted(adata.uns)`)
  mn_key in adata.uns_keys()


In [47]:
merged.obs.to_csv(result_folder + "/merged.obs.csv.gz", compression="gzip")
merged.var.to_csv(result_folder + "/merged.var.csv.gz", compression="gzip")

In [48]:
#write out peak memory at end
mem_usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
# print the memory usage in megabytes
print("Peak memory use in Gb  " + str(round(mem_usage / 1024 / 1024,2 )) + " PID  " + str(os.getpid()))

os.mkdir(os.path.join(result_folder, "Peak memory use in Gb " + str(round(mem_usage / 1024 / 1024 ,2))))

Peak memory use in Gb  225.17 PID  400


In [49]:
end_time = time.time()
print("Time taken h_m_s " + str(datetime.timedelta(seconds=end_time-start_time)).replace(':', '_').split('.')[0])
os.mkdir(os.path.join(result_folder, "Time taken h_m_s " + str(datetime.timedelta(seconds=end_time-start_time)).replace(':', '_').split('.')[0]))

Time taken h_m_s 1_44_44


In [50]:
print("Done!")

Done!


In [51]:
print(result_folder)

/sbgenomics/output-files/GEN_A3_PFC_vrs_GEN_A1-2.cortex.subclass.1780583145
